# A/B тест новой формы оплаты ЖКХ
**Гипотеза:** форма с автозаполнением реквизитов увеличит долю успешных оплат ЖКХ.
**Ключевая метрика:** конверсия из шага 1 (открыл форму) в шаг 4 (успешная оплата).
**Период:** 1–30 апреля 2024. **Единица рандомизации:** пользователь (клиент).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf

pd.set_option('display.width', 200)
FILE = 'Данные для кейса по ПА.xlsx'
STEPS = ['step1_opened', 'step2_entered', 'step3_confirmed', 'step4_success']

## 1. Загрузка и первичный осмотр

In [2]:
users_raw = pd.read_excel(FILE, sheet_name='Пользователи')
pay_raw = pd.read_excel(FILE, sheet_name='Платежи')
print(users_raw.shape, pay_raw.shape)
users_raw.head()

(1504, 5) (2661, 7)


,user_id,group,age,city,device_type
0,1001,A,31,Москва,ios
1,1002,A,31,Москва,ios
2,1003,A,25,Москва,android
3,1004,A,20,Москва,ios
4,1005,A,34,Москва,ios


In [3]:
print('Пропуски в users:'); print(users_raw.isna().sum(), '\n')
print('Возраст: min =', users_raw.age.min(), ', max =', users_raw.age.max())
print('Дубли user_id:', users_raw.user_id.duplicated().sum())
users_raw[users_raw.isna().any(axis=1) | ~users_raw.age.between(14, 100)]

Пропуски в users:
user_id        0
group          1
age            0
city           1
device_type    0
dtype: int64 

Возраст: min = -5 , max = 150
Дубли user_id: 0


,user_id,group,age,city,device_type
1500,9001,NaN,25,Москва,ios
1501,9002,A,-5,СПб,android
1502,9003,B,150,Регион,ios
1503,9004,A,30,NaN,android


In [4]:
print('Пропуски в payments:'); print(pay_raw.isna().sum(), '\n')
print('Дубли payment_id:', pay_raw.payment_id.duplicated().sum())

# платежи, где группа не совпадает с группой пользователя или пользователь неизвестен
check = pay_raw.merge(users_raw[['user_id', 'group']], on='user_id', how='left', suffixes=('', '_user'))
check[(check.group != check.group_user)]

Пропуски в payments:
payment_id            0
user_id               1
group                 0
step1_opened          0
step2_entered       405
step3_confirmed     942
step4_success      1139
dtype: int64 

Дубли payment_id: 0


,payment_id,user_id,group,step1_opened,step2_entered,step3_confirmed,step4_success,group_user
2659,99002,1010.0,B,2024-04-08 14:20:00,2024-04-08 14:23:00,2024-04-08 14:28:00,2024-04-08 14:29:00,A
2660,99003,NaN,A,2024-04-12 09:15:00,NaT,NaT,NaT,NaN


In [5]:
# логика воронки: нет пропущенных шагов и время идёт вперёд
for prev, nxt in zip(STEPS[:-1], STEPS[1:]):
    skipped = (pay_raw[nxt].notna() & pay_raw[prev].isna()).sum()
    minutes = (pay_raw[nxt] - pay_raw[prev]).dt.total_seconds() / 60
    print(f'{prev} -> {nxt}: пропущен шаг {skipped}, отрицательное время {(minutes < 0).sum()}, '
          f'мин/медиана/макс = {minutes.min():.0f}/{minutes.median():.0f}/{minutes.max():.0f}')

# записи с id вне основного диапазона
pay_raw[pay_raw.payment_id >= 99000]

step1_opened -> step2_entered: пропущен шаг 0, отрицательное время 0, мин/медиана/макс = 1/1/13
step2_entered -> step3_confirmed: пропущен шаг 0, отрицательное время 0, мин/медиана/макс = 1/4/11
step3_confirmed -> step4_success: пропущен шаг 0, отрицательное время 0, мин/медиана/макс = 1/6/9


,payment_id,user_id,group,step1_opened,step2_entered,step3_confirmed,step4_success
2658,99001,1005.0,A,2024-04-05 10:30:00,2024-04-05 10:33:00,2024-04-05 10:38:00,2024-04-05 10:39:00
2659,99002,1010.0,B,2024-04-08 14:20:00,2024-04-08 14:23:00,2024-04-08 14:28:00,2024-04-08 14:29:00
2660,99003,NaN,A,2024-04-12 09:15:00,NaT,NaT,NaT


**Что нашлось:**
- `user_id` 9001–9004 — технические записи вне основного диапазона id (1001–2500), у каждой дефект:
  9001 без группы, 9002 возраст −5, 9003 возраст 150, 9004 без города.
- `payment_id` 99001–99003 — вне диапазона id: 99003 без пользователя, 99002 пользователь 1010 из группы A
  получил платёж в группе B (пересечение групп), 99001 — подтверждение→оплата за 1 минуту при типичных 5–6.
- Пропущенных шагов воронки и обратного хода времени нет; NaT на шагах 2–4 — это нормальный отвал.

## 2. Очистка

In [6]:
users = users_raw[users_raw.user_id < 9000]                 # технические / некорректные пользователи
users = users[users.user_id != 1010]                        # видел обе формы — исключаем целиком

pay = pay_raw[(pay_raw.payment_id < 99000) & pay_raw.user_id.isin(users.user_id)].copy()
pay['user_id'] = pay.user_id.astype(int)

print(f'Пользователи: {len(users_raw)} -> {len(users)}')
print(f'Платежи: {len(pay_raw)} -> {len(pay)}')

Пользователи: 1504 -> 1499
Платежи: 2661 -> 2656


## 3. Корректность сплита

In [7]:
n_users = users.group.value_counts().sort_index()
print('Пользователей в группах:', n_users.to_dict())
print('SRM, хи-квадрат p =', round(stats.chisquare(n_users).pvalue, 3))   # p > 0.05 -> сплит 50/50 ок

print('Средний возраст:', users.groupby('group').age.mean().round(1).to_dict(),
      ' t-test p =', round(stats.ttest_ind(*[g.age for _, g in users.groupby('group')]).pvalue, 3))
for col in ['device_type', 'city']:
    table = pd.crosstab(users[col], users.group)
    print(col, (table / table.sum()).round(3).to_dict(), ' chi2 p =', round(stats.chi2_contingency(table)[1], 3))

Пользователей в группах: {'A': 749, 'B': 750}
SRM, хи-квадрат p = 0.979
Средний возраст: {'A': 32.8, 'B': 32.7}  t-test p = 0.807
device_type {'A': {'android': 0.541, 'ios': 0.459}, 'B': {'android': 0.492, 'ios': 0.508}}  chi2 p = 0.066
city {'A': {'Москва': 0.383, 'Регион': 0.358, 'СПб': 0.259}, 'B': {'Москва': 0.408, 'Регион': 0.327, 'СПб': 0.265}}  chi2 p = 0.427


Сплит 50/50 корректен, группы сбалансированы по возрасту и городу. По устройству небольшой перекос
(Android 54% в A против 49% в B, p = 0.07) — ниже проверим, что он не влияет на вывод.

## 4. Воронка по группам

In [8]:
funnel = pay.groupby('group')[STEPS].count()
display(funnel)
print('Доля от шага 1:'); display(funnel.div(funnel.step1_opened, axis=0).round(3))
print('Конверсия шаг -> шаг:'); display((funnel / funnel.shift(axis=1)).round(3))

,step1_opened,step2_entered,step3_confirmed,step4_success
group,,,,
A,1305,1059,756,649
B,1351,1194,960,870


Доля от шага 1:


,step1_opened,step2_entered,step3_confirmed,step4_success
group,,,,
A,1.0,0.811,0.579,0.497
B,1.0,0.884,0.711,0.644


Конверсия шаг -> шаг:


,step1_opened,step2_entered,step3_confirmed,step4_success
group,,,,
A,NaN,0.811,0.714,0.858
B,NaN,0.884,0.804,0.906


## 5. Основная метрика и статтест

Сплит был **по пользователям**, а у одного пользователя бывает до 4 попыток оплаты. Попытки одного
человека зависимы, поэтому обычный z-тест по попыткам занижает дисперсию. Поэтому:
1. **Конверсия попыток (шаг 1 → шаг 4)** — ratio-метрика, тестируем **дельта-методом** (дисперсия считается по пользователям).
2. **Проверка устойчивости:** доля пользователей с ≥1 успешной оплатой (z-тест для долей, наблюдения независимы) и бутстрап по пользователям.

In [9]:
users_agg = (pay.groupby(['user_id', 'group'])
             .agg(attempts=('payment_id', 'size'), success=('step4_success', 'count'))
             .reset_index()
             .merge(users[['user_id', 'age', 'city', 'device_type']], on='user_id'))
users_agg['paid'] = (users_agg.success > 0).astype(int)
A = users_agg[users_agg.group == 'A']
B = users_agg[users_agg.group == 'B']


def delta_ratio(df):
    x, y = df.success, df.attempts
    ratio = x.sum() / y.sum()
    var = (x.var() - 2 * ratio * np.cov(x, y)[0, 1] + ratio ** 2 * y.var()) / (y.mean() ** 2 * len(df))
    return ratio, var


conv_a, var_a = delta_ratio(A)
conv_b, var_b = delta_ratio(B)
se = np.sqrt(var_a + var_b)
z = (conv_b - conv_a) / se
print(f'Конверсия попыток: A = {conv_a:.1%}, B = {conv_b:.1%}')
print(f'Разница = {conv_b - conv_a:+.1%} [95% ДИ {conv_b - conv_a - 1.96 * se:+.1%}; {conv_b - conv_a + 1.96 * se:+.1%}], '
      f'uplift = {conv_b / conv_a - 1:+.1%}, p = {2 * stats.norm.sf(abs(z)):.1e}')

Конверсия попыток: A = 49.7%, B = 64.4%
Разница = +14.7% [95% ДИ +11.0%; +18.4%], uplift = +29.5%, p = 9.3e-15


In [10]:
paid = np.array([B.paid.sum(), A.paid.sum()])
total = np.array([len(B), len(A)])
z, p = proportions_ztest(paid, total)
low, high = confint_proportions_2indep(paid[0], total[0], paid[1], total[1], method='wald')
print(f'Доля заплативших: A = {A.paid.mean():.1%} ({paid[1]}/{total[1]}), B = {B.paid.mean():.1%} ({paid[0]}/{total[0]})')
print(f'Разница = {B.paid.mean() - A.paid.mean():+.1%} [95% ДИ {low:+.1%}; {high:+.1%}], p = {p:.1e}')

rng = np.random.default_rng(42)
def boot_conv(df, n=10_000):
    idx = rng.integers(0, len(df), (n, len(df)))
    return df.success.values[idx].sum(1) / df.attempts.values[idx].sum(1)
diff = boot_conv(B) - boot_conv(A)
print(f'Бутстрап, разница конверсии попыток: 95% ДИ [{np.percentile(diff, 2.5):+.1%}; {np.percentile(diff, 97.5):+.1%}]')

Доля заплативших: A = 65.6% (491/749), B = 80.4% (603/750)
Разница = +14.8% [95% ДИ +10.4%; +19.3%], p = 9.7e-11


Бутстрап, разница конверсии попыток: 95% ДИ [+10.9%; +18.4%]


Все три подхода дают одно: эффект ≈ +15 п.п., ДИ далеко от нуля, p < 0.001.

## 6. На каком шаге выигрывает новая форма

In [11]:
for prev, nxt in zip(STEPS[:-1], STEPS[1:]):
    reached = pay[pay[prev].notna()]
    rate = reached.groupby('group')[nxt].apply(lambda s: s.notna().mean())
    counts = reached.groupby('group')[nxt].agg(['count', 'size'])
    _, p = proportions_ztest(counts['count'], counts['size'])
    print(f'{prev:>16} -> {nxt:<16} A = {rate.A:.1%}  B = {rate.B:.1%}  разница = {rate.B - rate.A:+.1%}  p = {p:.1e}')

    step1_opened -> step2_entered    A = 81.1%  B = 88.4%  разница = +7.2%  p = 2.1e-07
   step2_entered -> step3_confirmed  A = 71.4%  B = 80.4%  разница = +9.0%  p = 5.4e-07
 step3_confirmed -> step4_success    A = 85.8%  B = 90.6%  разница = +4.8%  p = 2.1e-03


Самый большой прирост — на вводе и подтверждении реквизитов (+7 и +9 п.п.), то есть ровно там,
где работает автозаполнение. Плюс меньше отказов на финальном шаге (+5 п.п.) — вероятно, меньше ошибок в реквизитах.

## 7. Сегменты

In [12]:
users_agg['age_group'] = pd.cut(users_agg.age, [0, 25, 35, 45, 100], labels=['18-25', '26-35', '36-45', '46+'])
rows = []
for col in ['device_type', 'city', 'age_group']:
    for value, g in users_agg.groupby(col, observed=True):
        a, b = g[g.group == 'A'].paid, g[g.group == 'B'].paid
        _, p = proportions_ztest([b.sum(), a.sum()], [len(b), len(a)])
        rows.append({'сегмент': f'{col} = {value}', 'n_A': len(a), 'n_B': len(b),
                     'A': a.mean(), 'B': b.mean(), 'разница': b.mean() - a.mean(), 'p': p})
segments = pd.DataFrame(rows)
segments['p_holm'] = multipletests(segments.p, method='holm')[1]   # поправка на 9 сравнений
segments.round(3)

,сегмент,n_A,n_B,A,B,разница,p,p_holm
0,device_type = android,405,369,0.644,0.799,0.155,0.000,0.000
1,device_type = ios,344,381,0.669,0.808,0.140,0.000,0.000
2,city = Москва,287,306,0.624,0.810,0.187,0.000,0.000
3,city = Регион,268,245,0.668,0.763,0.095,0.017,0.045
4,city = СПб,194,199,0.686,0.844,0.159,0.000,0.001
5,age_group = 18-25,148,166,0.703,0.819,0.117,0.015,0.045
6,age_group = 26-35,341,306,0.633,0.807,0.174,0.000,0.000
7,age_group = 36-45,212,243,0.646,0.794,0.148,0.000,0.002
8,age_group = 46+,48,35,0.708,0.771,0.063,0.520,0.520


In [13]:
# контроль перекоса по устройству + проверка взаимодействия группа × устройство
model = smf.logit('paid ~ C(group) * C(device_type) + C(city) + age', data=users_agg).fit(disp=0)
model.summary().tables[1]

,coef,std err,z,P>|z|,[0.025,0.975]
Intercept,0.7298,0.274,2.660,0.008,0.192,1.267
C(group)[T.B],0.8018,0.167,4.803,0.000,0.475,1.129
C(device_type)[T.ios],0.1256,0.155,0.809,0.418,-0.178,0.430
C(city)[T.Регион],0.0018,0.136,0.013,0.990,-0.265,0.269
C(city)[T.СПб],0.2654,0.153,1.737,0.082,-0.034,0.565
C(group)[T.B]:C(device_type)[T.ios],-0.0787,0.241,-0.326,0.744,-0.551,0.394
age,-0.0064,0.008,-0.846,0.397,-0.021,0.008


Эффект положительный во всех сегментах и значим везде, кроме 46+ (там всего 83 человека).
Взаимодействие группа × устройство незначимо (p = 0.74) — форма одинаково работает на iOS и Android,
а с учётом перекоса по устройствам эффект группы B остаётся (p < 0.001).

## 8. Контрольные метрики и стабильность

In [14]:
pay['minutes_to_pay'] = (pay.step4_success - pay.step1_opened).dt.total_seconds() / 60
print('Медиана времени до оплаты, мин:', pay.groupby('group').minutes_to_pay.median().to_dict(),
      ' Mann-Whitney p =', round(stats.mannwhitneyu(*[g.minutes_to_pay.dropna() for _, g in pay.groupby('group')]).pvalue, 2))

users_agg['fails'] = users_agg.attempts - users_agg.success
for col in ['attempts', 'success', 'fails']:
    a, b = users_agg[users_agg.group == 'A'][col], users_agg[users_agg.group == 'B'][col]
    print(f'{col:>8} на пользователя: A = {a.mean():.2f}, B = {b.mean():.2f}, MW p = {stats.mannwhitneyu(a, b).pvalue:.3f}')

pay['week'] = ((pay.step1_opened.dt.day - 1) // 7 + 1).clip(upper=4)
pay.groupby(['week', 'group']).step4_success.apply(lambda s: s.notna().mean()).unstack().round(3)

Медиана времени до оплаты, мин: {'A': 12.0, 'B': 12.0}  Mann-Whitney p = 0.39
attempts на пользователя: A = 1.74, B = 1.80, MW p = 0.150
 success на пользователя: A = 0.87, B = 1.16, MW p = 0.000
   fails на пользователя: A = 0.88, B = 0.64, MW p = 0.000


group,A,B
week,,
1,0.507,0.663
2,0.523,0.660
3,0.480,0.623
4,0.484,0.629


- Время до оплаты не изменилось (медиана 12 мин в обеих группах).
- Попыток на пользователя столько же (p = 0.15), но **неудачных попыток меньше** (0.88 → 0.64), а **успешных больше** (0.87 → 1.16).
- Эффект держится все 4 недели — это не эффект новизны.

## 9. Вывод
Новая форма повышает конверсию попыток из открытия в успешную оплату с **49.7% до 64.4% (+14.7 п.п., +29.5%)**,
доля пользователей, оплативших хотя бы раз, растёт с **65.6% до 80.4%**. Результат статистически значим,
устойчив к методу, одинаков по сегментам и стабилен во времени. **Рекомендация: раскатывать на всех пользователей.**